**Sampler Demo - Running on a Fake Backend Simulator.**

This notebook builds the $\Psi^+$ Bell state on two qubits and runs it on
`FakeAlmadenV2`, a local simulator that mimics the noise model and gate set
of a real IBM device. No IBM account or cloud queue is needed.

The difference is which primitive we use, and that changes what comes back:

- The **Estimator** answers "what is the average value of an observable?"
  It hides the individual shots and reports expectation values.
- The **Sampler** answers "what did each shot actually measure?"
  It returns the raw bitstrings, which we tally into counts.

Because we want the raw shot data, this circuit must include measurement
gates - the Estimator version did not need them.
We let the runtime pick the number of shots, and the histogram at the end
shows how many of those shots landed on each outcome.

---
**Cell 01 - Build the Psi+ Bell state circuit.**

A Hadamard gate on qubit 0 followed by a CNOT (control 0, target 1) produces
the $\Psi^+$ Bell state $\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle)$.
The two qubits are entangled: neither has a definite value on its own, but
they always agree when measured.

`measure_all()` adds a classical register and measures every qubit into it.
This is the one structural change from the Estimator notebook.
The Sampler reports what the classical bits held after each shot, so without
measurements there would be nothing to report.

In [ ]:
"""sampler_simulator.ipynb"""

# Cell 01 - Create a Psi+ Bell State (|00> + |11>)

from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

---
**Cell 02 - Transpile and run on the fake backend.**

`FakeAlmadenV2` is a local fake backend: it reproduces the native gate set,
qubit connectivity, and calibrated noise model of the real Almaden device
but runs entirely on your machine with no cloud connection or queue wait.

The pass manager rewrites our two-gate circuit into the backend's native
gates and maps our logical qubits onto physical ones that are actually
connected on the chip. The result is called an ISA circuit
(Instruction Set Architecture), and it is what the backend can really run.

We do not tell the Sampler how many shots to run, so it uses its default.
Each shot is one full run of the circuit ending in a measurement, so the
number of shots is simply how many bitstrings we get back - and the
histogram in Cell 04 shows exactly how they were distributed.

In [ ]:
# Cell 02 - Transpile and run on the fake backend

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeAlmadenV2

backend = FakeAlmadenV2()
config = backend.configuration()
print(
    f"{config.backend_name:15}: Qubits = {config.n_qubits}: Gates = {config.basis_gates}"
)

# Transpile the circuit for the fake backend's native gate set and connectivity
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
display(isa_circuit.draw("mpl", idle_wires=False))

# Run the circuit, letting the runtime choose the number of shots
sampler = Sampler(backend)
job = sampler.run([isa_circuit])
print(f"\nJob ID: {job.job_id()}")

---
**Cell 03 - Retrieve the actual counts.**

Because the fake backend runs locally, `job.result()` returns immediately
rather than blocking on a cloud queue.

The result for our single circuit (a single PUB, or Primitive Unified Bloc)
carries a data field named after the circuit's classical register.
`measure_all()` names that register `meas`, so the bitstrings live in
`pub_result.data.meas`. Calling `get_counts()` tallies them into a
dictionary mapping each bitstring to how many shots produced it.

For an ideal $\Psi^+$ Bell state you would see only `00` and `11`, each about
half the shots. The fake backend's noise model will also produce a small
number of `01` and `10` results - the two qubits disagreeing, which the ideal
state forbids. These come from readout and gate errors, and they are exactly
what real hardware gives you.

In [ ]:
# Cell 03 - Retrieve the actual counts

pub_result = job.result()[0]

# measure_all() names the classical register "meas"
counts = pub_result.data.meas.get_counts()
total_shots = sum(counts.values())

print(f"Total shots actually run: {total_shots}\n")
print(f"{'Bitstring':>10}{'Counts':>10}{'Percent':>10}")
for bitstring in sorted(counts):
    shots = counts[bitstring]
    print(f"{bitstring:>10}{shots:>10}{shots / total_shots:>10.2%}")

---
**Cell 04 - Plot the counts as a histogram.**

The table above is the raw answer, but a histogram makes the structure
obvious at a glance: two tall bars at `00` and `11` of nearly equal height,
and two short error bars at `01` and `10`.

The two tall bars are the entanglement. The two short bars are the noise.
This single picture is what the Estimator compressed down to
$\langle ZZ \rangle \approx 1$ in the companion notebook - same experiment,
but here you can see every shot that went into it.

In [ ]:
# Cell 04 - Plot the counts as a histogram

from qiskit.visualization import plot_histogram

# plot_histogram builds and returns its own figure, so let the cell
# display that figure directly rather than calling plt.show()
plot_histogram(counts, color="steelblue", title="Psi+ Bell State Counts (Fake Backend)")